In [ ]:
import time
from collections.abc import Sequence
from dataclasses import dataclass
from datetime import datetime
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from torchinfo import summary
from torchvision.datasets import CIFAR10

from mlkit.blocks.embeddings import SinusoidalEmbedding, TimeEmbeddingConfig
from mlkit.blocks.unet import UNetBlockConfig
from mlkit.core.callbacks import MLflowLogger, ProgressLogger
from mlkit.core.history import History
from mlkit.core.trainer import Trainer
from mlkit.data.splitter import StratifiedSplitter
from mlkit.models.unet import UNet, UNetConfig
from mlkit.tasks.diffusion import DiffusionTask, LinearSchedule, NoiseSchedule
from mlkit.utils.device import get_device
from mlkit.utils.logs import setup_logging
from mlkit.utils.seed import seed_everything
from mlkit.utils.tracking import flatten_params, setup_mlflow
from mlkit.visual.curves import plot_comparison
from mlkit.visual.images import plot_image_grid
from mlkit.visual.saver import FigureSaver
from mlkit.visual.style import use_style

import mlflow

setup_logging()
use_style()

ROOT = Path("../..")
DATA_DIR = ROOT / "datasets/raw"
setup_mlflow(ROOT, "cgw_task_c")

SEED = 42
IMAGE_SHAPE = (3, 32, 32)
ANIMAL_CLASSES = ("bird", "cat", "deer", "dog", "frog", "horse")
VAL_FRACTION = 0.1

STEPS = 1000
BETA_START = 1e-4
BETA_END = 0.02

IMAGE_INDEX = 7
MSE_EXAMPLE_VALUES = 4

EMBEDDING_DIM = 64
TIME_DIM = 256
DROPOUT = 0.1
VARIANTS = {
    "base": (32, 64, 128),
    "wide": (64, 128, 256),
}

BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 2e-4
TIMING_WARMUP = 5
TIMING_STEPS = 50

TIME_DEMO_STEPS = (100, 500, 900)
TIME_DEMO_BATCH = 256
STEP_COUNTS = (100, 500, 1000)
SAMPLING_BATCH = 16

seed_everything(SEED)
device = get_device()

RUN_NAME = f"task_c_{datetime.now():%Y%m%d_%H%M%S}"
saver = FigureSaver(Path("artifacts") / RUN_NAME)

In [ ]:
train_raw = CIFAR10(DATA_DIR, train=True, download=True)
test_raw = CIFAR10(DATA_DIR, train=False, download=True)
animal_labels = {train_raw.class_to_idx[name] for name in ANIMAL_CLASSES}


def to_model_range(data: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(data).permute(0, 3, 1, 2).float() / 127.5 - 1


def to_display(x: torch.Tensor) -> torch.Tensor:
    return ((x + 1) / 2).clamp(0, 1).cpu()


def animal_subset(dataset: CIFAR10) -> tuple[torch.Tensor, list[int]]:
    indices = [index for index, label in enumerate(dataset.targets) if label in animal_labels]
    return to_model_range(dataset.data[indices]), [dataset.targets[index] for index in indices]


train_images, train_targets = animal_subset(train_raw)
test_images, _ = animal_subset(test_raw)
fit_indices, val_indices = StratifiedSplitter((1 - VAL_FRACTION, VAL_FRACTION), seed=SEED).split(train_targets)
fit_images, val_images = train_images[fit_indices], train_images[val_indices]

train_loader = DataLoader(fit_images, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_images, batch_size=2 * BATCH_SIZE, pin_memory=True)
test_loader = DataLoader(test_images, batch_size=2 * BATCH_SIZE, pin_memory=True)

image_class = train_raw.classes[train_raw.targets[IMAGE_INDEX]]
if image_class not in ANIMAL_CLASSES:
    raise ValueError(f"IMAGE_INDEX must point to one of {ANIMAL_CLASSES}, got {IMAGE_INDEX} ({image_class})")
x0 = to_model_range(train_raw.data[IMAGE_INDEX : IMAGE_INDEX + 1])

print(f"Animals: train {len(fit_images)} | val {len(val_images)} | test {len(test_images)} | image {tuple(fit_images.shape[1:])}")
print(f"Range: [{fit_images.min().item():.1f}, {fit_images.max().item():.1f}] | x0: {image_class}, shape {tuple(x0.shape)}")

In [ ]:
schedule = NoiseSchedule(STEPS, LinearSchedule(BETA_START, BETA_END))
generator = torch.Generator().manual_seed(SEED)
t = torch.randint(STEPS, (1,), generator=generator)
eps = torch.randn(x0.shape, generator=generator)
x_t = schedule.add_noise(x0, t, eps)

print(f"Random step t = {t.item()} | alpha_bar_t = {schedule.alpha_bars[t].item():.4f}")
print(f"U-Net input: x_t {tuple(x_t.shape)}, t {tuple(t.shape)} {t.dtype} | target: eps {tuple(eps.shape)}")
titles = [f"x0: {image_class}", "eps (target)", f"x_t, t = {t.item()}"]
saver.save(plot_image_grid(to_display(torch.cat([x0, eps, x_t])), titles, ncols=len(titles)), "unet_input");

In [ ]:
sinusoidal = SinusoidalEmbedding(EMBEDDING_DIM)
embeddings = sinusoidal(torch.arange(STEPS))
similarity = F.cosine_similarity(embeddings[:, None], embeddings[None], dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
image = axes[0].imshow(embeddings.T.numpy(), aspect="auto", cmap="RdBu", interpolation="nearest")
axes[0].set_xlabel("Step t")
axes[0].set_ylabel("Embedding dimension")
axes[0].set_title("Sinusoidal time embedding")
fig.colorbar(image, ax=axes[0])
image = axes[1].imshow(similarity.numpy(), cmap="viridis")
axes[1].set_xlabel("Step t")
axes[1].set_ylabel("Step t")
axes[1].set_title("Cosine similarity between embeddings")
fig.colorbar(image, ax=axes[1])
for ax in axes:
    ax.grid(False)
fig.tight_layout()
saver.save(fig, "time_embedding")

print(f"t {tuple(t.shape)} -> sinusoidal {tuple(sinusoidal(t).shape)} -> Linear, SiLU, Linear -> ({t.numel()}, {TIME_DIM})")
print("In every U-Net block: SiLU -> Linear(time_dim, channels) -> added to the feature map of each pixel")

In [ ]:
def unet_config(channels: Sequence[int]) -> UNetConfig:
    return UNetConfig(
        in_channels=IMAGE_SHAPE[0],
        input_size=IMAGE_SHAPE[1:],
        blocks=tuple(UNetBlockConfig(width, dropout=DROPOUT) for width in channels),
        time=TimeEmbeddingConfig(EMBEDDING_DIM, TIME_DIM),
    )


seed_everything(SEED)
model = UNet(unet_config(VARIANTS["wide"])).eval()
print(model)

with torch.no_grad():
    eps_pred = model(x_t, t)
print(f"Input {tuple(x_t.shape)} -> output {tuple(eps_pred.shape)}")

In [ ]:
target_values = eps.flatten()[:MSE_EXAMPLE_VALUES]
pred_values = eps_pred.flatten()[:MSE_EXAMPLE_VALUES]
squared = (target_values - pred_values) ** 2

for index, (target, pred, value) in enumerate(zip(target_values.tolist(), pred_values.tolist(), squared.tolist())):
    print(f"value {index}: eps = {target:.4f} | eps_pred = {pred:.4f} | (eps - eps_pred)^2 = {value:.4f}")

manual = sum(squared.tolist()) / MSE_EXAMPLE_VALUES
print(f"Manual MSE = ({' + '.join(f'{value:.4f}' for value in squared.tolist())}) / {MSE_EXAMPLE_VALUES} = {manual:.4f}")
print(f"F.mse_loss on the same {MSE_EXAMPLE_VALUES} values: {F.mse_loss(pred_values, target_values).item():.4f}")

untrained_loss = F.mse_loss(eps_pred, eps).item()
print(f"All {eps.numel()} values: mean((eps - eps_pred)^2) = {torch.mean((eps - eps_pred) ** 2).item():.4f} | F.mse_loss = {untrained_loss:.4f}")

In [ ]:
stats = summary(
    model,
    input_data=[x_t, t],
    depth=3,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    device="cpu",
    verbose=0,
)
print(stats)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: total {total:,} | trainable {trainable:,} | torchinfo: {stats.total_params:,} / {stats.trainable_params:,}")

conv_layers = {
    "stem": model.stem[0],
    "encoder 2, conv1": model.encoder[1].conv1[-1],
    "downsampling 1": model.downs[0][0],
    "head": model.head[-1],
}
display(pd.DataFrame([
    {
        "layer": name,
        "kernel": f"{conv.kernel_size[0]}x{conv.kernel_size[1]}",
        "in channels": conv.in_channels,
        "filters": conv.out_channels,
        "bias": conv.bias is not None,
        "manual: (k*k*C_in + 1) * filters": (conv.kernel_size[0] * conv.kernel_size[1] * conv.in_channels + 1) * conv.out_channels,
        "PyTorch": sum(p.numel() for p in conv.parameters()),
    }
    for name, conv in conv_layers.items()
]).set_index("layer"))

stages: dict[str, nn.Module] = {"stem": model.stem}
for index, (block, down) in enumerate(zip(model.encoder, model.downs), start=1):
    stages |= {f"encoder {index}": block, f"down {index}": down}
stages["bottleneck"] = model.bottleneck
for index, (up, block) in enumerate(zip(model.ups, model.decoder), start=1):
    stages |= {f"up {index}": up, f"decoder {index}": block}
stages["head"] = model.head

shapes: dict[str, tuple[int, ...]] = {}
handles = [
    module.register_forward_hook(lambda module, inputs, output, name=name: shapes.__setitem__(name, tuple(output.shape)))
    for name, module in stages.items()
]
with torch.no_grad():
    model(x_t, t)
for handle in handles:
    handle.remove()

pd.DataFrame(
    {
        "channels": [shape[1] for shape in shapes.values()],
        "resolution": [f"{shape[2]}x{shape[3]}" for shape in shapes.values()],
        "parameters": [sum(p.numel() for p in stages[name].parameters()) for name in shapes],
    },
    index=list(shapes),
)

In [ ]:
@dataclass
class RunResult:
    task: DiffusionTask
    history: History
    parameters: int
    parameter_mb: float
    peak_memory_mb: float
    step_ms: float
    seconds: float


def synchronize() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()


def make_task(channels: Sequence[int], steps: int = STEPS) -> DiffusionTask:
    return DiffusionTask(
        model=UNet(unet_config(channels)),
        optimizer=partial(torch.optim.Adam, lr=LEARNING_RATE),
        schedule=NoiseSchedule(steps, LinearSchedule(BETA_START, BETA_END)),
        image_shape=IMAGE_SHAPE,
    )


def benchmark_training_step(task: DiffusionTask) -> tuple[float, float]:
    task.to(device).train()
    batches = iter(train_loader)
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    for _ in range(TIMING_WARMUP):
        task.training_step(next(batches).to(device))
    synchronize()
    start = time.perf_counter()
    for _ in range(TIMING_STEPS):
        task.training_step(next(batches).to(device))
    synchronize()

    step_ms = (time.perf_counter() - start) / TIMING_STEPS * 1000
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else float("nan")
    return step_ms, peak_mb


def run(name: str, channels: Sequence[int]) -> RunResult:
    seed_everything(SEED)
    step_ms, peak_mb = benchmark_training_step(make_task(channels))

    seed_everything(SEED)
    task = make_task(channels)
    parameters = sum(p.numel() for p in task.model.parameters())
    parameter_mb = sum(p.numel() * p.element_size() for p in task.model.parameters()) / 2**20

    with mlflow.start_run(run_name=name):
        mlflow.log_params(
            flatten_params(task.model.config)
            | {"steps": STEPS, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "epochs": EPOCHS}
        )
        trainer = Trainer(device=device, callbacks=[ProgressLogger(), MLflowLogger()])
        start = time.perf_counter()
        history = trainer.fit(task, train_loader, val_loader, epochs=EPOCHS)
        seconds = time.perf_counter() - start
        mlflow.log_metrics({
            "parameters": parameters,
            "parameter_mb": parameter_mb,
            "peak_memory_mb": peak_mb,
            "step_ms": step_ms,
            "train_seconds": seconds,
        })

    task.eval()
    return RunResult(task, history, parameters, parameter_mb, peak_mb, step_ms, seconds)

In [ ]:
results: dict[str, RunResult] = {}
for name, channels in VARIANTS.items():
    print(f"===== {name}: channels {channels}")
    results[name] = run(name, channels)

In [ ]:
def example_loss(model: nn.Module) -> float:
    with torch.no_grad():
        return F.mse_loss(model(x_t.to(device), t.to(device)), eps.to(device)).item()


histories = {name: result.history for name, result in results.items()}
saver.save(plot_comparison(histories, ["loss"], splits=("train", "val")), "loss")

configs = pd.DataFrame.from_dict(
    {
        name: {
            "channels": str(VARIANTS[name]),
            "parameters": result.parameters,
            "weights, MB": result.parameter_mb,
            "weights + grads + Adam states, MB": 4 * result.parameter_mb,
            "peak GPU memory per training step, MB": result.peak_memory_mb,
            "time per training step, ms": result.step_ms,
            "time per epoch, s": result.seconds / EPOCHS,
            "train loss (last epoch)": result.history.last("train/loss"),
            "val loss (last epoch)": result.history.last("val/loss"),
            "val loss (best)": result.history.best("val/loss", "min")[1],
            "loss on the c.1 example": example_loss(result.task.model),
        }
        for name, result in results.items()
    },
    orient="index",
)
configs.to_csv(saver.directory / "configs.csv")

print(f"Loss on the c.1 example before training (wide): {untrained_loss:.4f}")
configs.round(4)

In [ ]:
trained = results["wide"].task
demo_images = val_images[:TIME_DEMO_BATCH].to(device)
demo_noise = torch.randn(demo_images.shape, generator=torch.Generator().manual_seed(SEED)).to(device)


def steps_like(images: torch.Tensor, step: int) -> torch.Tensor:
    return torch.full((len(images),), step, dtype=torch.long, device=images.device)


eps_errors: dict[str, dict[str, float]] = {}
with torch.no_grad():
    for true_step in TIME_DEMO_STEPS:
        noisy = trained.schedule.add_noise(demo_images, steps_like(demo_images, true_step), demo_noise)
        eps_errors[f"true t = {true_step}"] = {
            f"given t = {given_step}": F.mse_loss(trained.model(noisy, steps_like(noisy, given_step)), demo_noise).item()
            for given_step in TIME_DEMO_STEPS
        }

    horse_step = TIME_DEMO_STEPS[len(TIME_DEMO_STEPS) // 2]
    horse = x0.to(device)
    horse_noisy = trained.schedule.add_noise(horse, steps_like(horse, horse_step), eps.to(device))
    estimates = [
        trained.schedule.predict_x0(
            horse_noisy, steps_like(horse, horse_step), trained.model(horse_noisy, steps_like(horse, given_step))
        ).clamp(-1, 1)
        for given_step in TIME_DEMO_STEPS
    ]

titles = ["x0", f"x_t, t = {horse_step}"] + [f"x0_hat, given t = {step}" for step in TIME_DEMO_STEPS]
saver.save(plot_image_grid(to_display(torch.cat([horse, horse_noisy, *estimates])), titles, ncols=len(titles)), "time_conditioning")

pd.DataFrame(eps_errors).T.round(4)

In [ ]:
def sample(model: nn.Module, schedule: NoiseSchedule, z: torch.Tensor, generator: torch.Generator) -> tuple[torch.Tensor, int]:
    x = z
    passes = 0
    with torch.no_grad():
        for step in range(schedule.steps - 1, -1, -1):
            eps_pred = model(x, torch.full((x.size(0),), step, dtype=torch.long, device=x.device))
            passes += 1
            noise = None if step == 0 else torch.randn(x.shape, generator=generator).to(x.device)
            x = schedule.step(x, step, eps_pred, noise)
    return x, passes


sampling_z = torch.randn(SAMPLING_BATCH, *IMAGE_SHAPE, generator=torch.Generator().manual_seed(SEED)).to(device)
records = []
for steps in STEP_COUNTS:
    step_schedule = NoiseSchedule(steps, LinearSchedule(BETA_START, BETA_END)).to(device)

    synchronize()
    start = time.perf_counter()
    _, passes = sample(trained.model, step_schedule, sampling_z, torch.Generator().manual_seed(SEED))
    synchronize()
    seconds = time.perf_counter() - start

    seed_everything(SEED)
    step_ms, _ = benchmark_training_step(make_task(VARIANTS["wide"], steps))

    records.append({
        "T": steps,
        "alpha_bar_T": step_schedule.alpha_bars[-1].item(),
        "U-Net passes per generated image": passes,
        "U-Net passes per image per training step": 1,
        f"sampling time for {SAMPLING_BATCH} images, s": seconds,
        "time per U-Net pass, ms": seconds / passes * 1000,
        "time per training step, ms": step_ms,
    })

steps_table = pd.DataFrame(records).set_index("T")
steps_table.to_csv(saver.directory / "diffusion_steps.csv")
steps_table.round(4)